# 🏦 Detecção de Fraude em Contas Bancárias — Benchmark de Modelos

**Dataset:** Bank Account Fraud (BAF) — NeurIPS 2022  
**Objetivo:** Comparar múltiplas famílias de modelos de Machine Learning na tarefa de detecção de fraude, avaliando tanto **desempenho preditivo** quanto **justiça algorítmica** (fairness).

### Modelos Avaliados

| Família | Modelos |
|---------|---------|
| **Lineares** | Logistic Regression, SGD Classifier |
| **Agregação (Ensemble Bagging)** | Random Forest, Extra Trees |
| **SVM** | LinearSVC (com calibração) |
| **Redes Neurais Leves** | MLP (scikit-learn) |
| **Redes Neurais Densas** | Keras/TensorFlow (arquitetura profunda) |

### Métricas de Avaliação
- **Desempenho:** Accuracy, Precision, Recall, F1-Score, AUC-ROC, AUC-PR
- **Fairness:** Equalized Odds Difference, Demographic Parity Difference (antes e depois de pós-processamento)

---

## 1. Instalação de Dependências e Imports

In [ ]:
# Instalação (descomente se necessário)
# !pip install pandas numpy scikit-learn matplotlib seaborn tensorflow imbalanced-learn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

# Configuração geral
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Imports carregados com sucesso!")
print(f"TensorFlow version: {tf.__version__}")

## 2. Carregamento dos Dados

In [ ]:
import os

# Tente carregar o dataset - ajuste o caminho conforme necessário
DATA_PATHS = [
    'Base.csv',
    'data/Base.csv',
    '../data/Base.csv',
    '/content/Base.csv',  # Google Colab
]

df = None
for path in DATA_PATHS:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Dataset carregado de: {path}")
        break

if df is None:
    print("Dataset nao encontrado nos caminhos padrao.")
    print("Faca o download do BAF dataset e ajuste o caminho:")
    print("  https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022")
    print()
    print("Exemplo: df = pd.read_csv('caminho/para/Base.csv')")
else:
    print(f"Shape: {df.shape}")
    print(f"Colunas: {df.columns.tolist()}")

> **Nota:** Se o dataset não foi encontrado, faça o download em:  
> [Kaggle - Bank Account Fraud Dataset (NeurIPS 2022)](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022)  
> E carregue com `df = pd.read_csv('caminho/Base.csv')` na célula acima.

## 3. Análise Exploratória de Dados (EDA)

Vamos explorar a distribuição das features, o desbalanceamento de classes, correlações e as distribuições por grupo protegido.

### 3.1 Visão Geral do Dataset

In [ ]:
print("=" * 60)
print("VISAO GERAL DO DATASET")
print("=" * 60)

print(f"\nDimensoes: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"\nPrimeiras linhas:")
df.head()

In [ ]:
print("Tipos de dados e valores nulos:\n")
info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'Nao-Nulos': df.count(),
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'Unicos': df.nunique()
})
info_df

In [ ]:
print("Estatisticas descritivas (variaveis numericas):\n")
df.describe().round(3)

### 3.2 Distribuição da Variável Alvo (fraud_bool)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target_counts = df['fraud_bool'].value_counts()
colors = ['#2ecc71', '#e74c3c']
labels = ['Legitima (0)', 'Fraude (1)']

axes[0].bar(labels, target_counts.values, color=colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + len(df)*0.005, f'{v:,}', ha='center', fontweight='bold', fontsize=12)
axes[0].set_title('Distribuicao de Classes', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Contagem')

axes[1].pie(target_counts.values, labels=labels, colors=colors, autopct='%1.2f%%',
            startangle=90, textprops={'fontsize': 12}, explode=[0, 0.05], shadow=True)
axes[1].set_title('Proporcao de Classes', fontsize=14, fontweight='bold')

plt.suptitle('Desbalanceamento de Classes no BAF Dataset', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

ratio = target_counts[0] / target_counts[1]
print(f"\nRazao de desbalanceamento: {ratio:.1f}:1 (legitimas : fraudes)")

### 3.3 Distribuição das Variáveis Numéricas

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_plot = [c for c in num_cols if c != 'fraud_bool']

n_cols_grid = 4
n_rows_grid = (len(num_cols_plot) + n_cols_grid - 1) // n_cols_grid

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(20, 4 * n_rows_grid))
axes = axes.flatten()

for i, col in enumerate(num_cols_plot):
    ax = axes[i]
    df[df['fraud_bool'] == 0][col].hist(bins=50, alpha=0.6, color='#2ecc71', label='Legitima', ax=ax, density=True)
    df[df['fraud_bool'] == 1][col].hist(bins=50, alpha=0.6, color='#e74c3c', label='Fraude', ax=ax, density=True)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuicao das Variaveis Numericas por Classe', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.4 Variáveis Categóricas

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Variaveis categoricas encontradas: {cat_cols}\n")

if len(cat_cols) > 0:
    n_cat = len(cat_cols)
    fig, axes = plt.subplots(1, min(n_cat, 4), figsize=(5 * min(n_cat, 4), 5))
    if n_cat == 1:
        axes = [axes]
    
    for i, col in enumerate(cat_cols[:4]):
        ax = axes[i]
        ct = pd.crosstab(df[col], df['fraud_bool'], normalize='index')
        ct.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black')
        ax.set_title(f'Taxa de Fraude por {col}', fontsize=11, fontweight='bold')
        ax.set_ylabel('Proporcao')
        ax.legend(['Legitima', 'Fraude'], fontsize=8)
        ax.tick_params(labelrotation=45, labelsize=8)
    
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma variavel categorica do tipo object/category encontrada.")

### 3.5 Matriz de Correlação

In [ ]:
corr_matrix = df[num_cols].corr()
target_corr = corr_matrix['fraud_bool'].drop('fraud_bool').abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

top_features = target_corr.head(15).index.tolist() + ['fraud_bool']
sns.heatmap(df[top_features].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0], square=True, linewidths=0.5, annot_kws={'size': 8})
axes[0].set_title('Correlacao - Top 15 Features vs Fraude', fontsize=13, fontweight='bold')

axes[1].barh(target_corr.head(20).index, target_corr.head(20).values, color='#3498db', edgecolor='black')
axes[1].set_xlabel('|Correlacao| com fraud_bool')
axes[1].set_title('Top 20 Correlacoes com a Variavel Alvo', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### 3.6 Análise do Atributo Protegido (Fairness)

In [ ]:
# No BAF dataset, o atributo sensivel principal e 'customer_age'
# Grupo protegido: jovens (age <= 2, discretizado) vs nao-jovens
SENSITIVE_FEATURE = 'customer_age_bin'

if 'customer_age' in df.columns:
    df['customer_age_bin'] = (df['customer_age'] <= 2).astype(int)
    print("Atributo protegido criado: customer_age_bin")
    print("  0 = Grupo nao-protegido (idade > jovem)")
    print("  1 = Grupo protegido (jovem)")
elif 'age' in df.columns:
    df['customer_age_bin'] = (df['age'] <= 30).astype(int)

if SENSITIVE_FEATURE in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    group_fraud = df.groupby(SENSITIVE_FEATURE)['fraud_bool'].mean()
    group_count = df.groupby(SENSITIVE_FEATURE)['fraud_bool'].count()
    colors_grp = ['#3498db', '#e67e22']
    
    axes[0].bar(group_count.index.astype(str), group_count.values, color=colors_grp, edgecolor='black')
    axes[0].set_title('Tamanho dos Grupos', fontsize=13, fontweight='bold')
    axes[0].set_xlabel(SENSITIVE_FEATURE)
    axes[0].set_ylabel('Contagem')
    for i, v in enumerate(group_count.values):
        axes[0].text(i, v + len(df)*0.005, f'{v:,}', ha='center', fontweight='bold')
    
    axes[1].bar(group_fraud.index.astype(str), group_fraud.values * 100, color=colors_grp, edgecolor='black')
    axes[1].set_title('Taxa de Fraude por Grupo', fontsize=13, fontweight='bold')
    axes[1].set_xlabel(SENSITIVE_FEATURE)
    axes[1].set_ylabel('Taxa de Fraude (%)')
    for i, v in enumerate(group_fraud.values):
        axes[1].text(i, v*100 + 0.1, f'{v*100:.2f}%', ha='center', fontweight='bold')
    
    plt.suptitle('Analise do Atributo Protegido para Fairness', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Pré-processamento

1. Encoding de variáveis categóricas (Label Encoding)
2. Separação treino/teste estratificada
3. Normalização (StandardScaler)
4. Separação do atributo sensível para fairness

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"  LabelEncoded: {col}")

print(f"\n{len(cat_cols)} colunas categoricas codificadas.")

In [ ]:
TARGET = 'fraud_bool'
SENSITIVE = 'customer_age_bin'

feature_cols = [c for c in df.columns if c not in [TARGET, SENSITIVE]]
X = df[feature_cols].copy()
y = df[TARGET].copy()

if SENSITIVE in df.columns:
    sensitive_attr = df[SENSITIVE].copy()
else:
    sensitive_attr = pd.Series(np.zeros(len(df)), name=SENSITIVE)

print(f"Features: {X.shape[1]} colunas")
print(f"Target: {y.value_counts().to_dict()}")
print(f"Atributo sensivel: {SENSITIVE}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

sens_train = sensitive_attr.loc[X_train.index]
sens_test = sensitive_attr.loc[X_test.index]

print(f"Treino: {X_train.shape[0]:,} amostras")
print(f"Teste:  {X_test.shape[0]:,} amostras")
print(f"Distribuicao treino - fraude: {y_train.mean():.4f}")
print(f"Distribuicao teste  - fraude: {y_test.mean():.4f}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("StandardScaler aplicado.")
print(f"   Media (treino, amostra): {X_train_scaled.mean(axis=0)[:3].round(4)}")
print(f"   Std   (treino, amostra): {X_train_scaled.std(axis=0)[:3].round(4)}")

## 5. Funções Auxiliares de Avaliação e Fairness

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name="Modelo"):
    metrics = {
        'Modelo': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'AUC-ROC': roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
        'AUC-PR': average_precision_score(y_true, y_prob) if y_prob is not None else np.nan,
    }
    return metrics


def compute_fairness_metrics(y_true, y_pred, sensitive, model_name="Modelo"):
    groups = np.unique(sensitive)
    if len(groups) < 2:
        return {'Modelo': model_name, 'Demographic Parity Diff': np.nan,
                'Equalized Odds Diff (TPR)': np.nan, 'Equalized Odds Diff (FPR)': np.nan}
    
    g0_mask = sensitive == groups[0]
    g1_mask = sensitive == groups[1]
    
    dp_g0 = y_pred[g0_mask].mean()
    dp_g1 = y_pred[g1_mask].mean()
    dp_diff = abs(dp_g0 - dp_g1)
    
    tpr_g0 = y_pred[(g0_mask) & (y_true == 1)].mean() if (g0_mask & (y_true == 1)).sum() > 0 else 0
    tpr_g1 = y_pred[(g1_mask) & (y_true == 1)].mean() if (g1_mask & (y_true == 1)).sum() > 0 else 0
    eo_tpr = abs(tpr_g0 - tpr_g1)
    
    fpr_g0 = y_pred[(g0_mask) & (y_true == 0)].mean() if (g0_mask & (y_true == 0)).sum() > 0 else 0
    fpr_g1 = y_pred[(g1_mask) & (y_true == 0)].mean() if (g1_mask & (y_true == 0)).sum() > 0 else 0
    eo_fpr = abs(fpr_g0 - fpr_g1)
    
    return {'Modelo': model_name, 'Demographic Parity Diff': dp_diff,
            'Equalized Odds Diff (TPR)': eo_tpr, 'Equalized Odds Diff (FPR)': eo_fpr}


def fairness_postprocess(y_prob, sensitive, y_true, threshold=0.5):
    groups = np.unique(sensitive)
    y_pred_fair = np.zeros_like(y_prob, dtype=int)
    
    for g in groups:
        mask = sensitive == g
        target_rate = y_true.mean()
        probs_g = y_prob[mask]
        sorted_probs = np.sort(probs_g)[::-1]
        n_positive = int(target_rate * len(probs_g))
        if n_positive > 0 and n_positive < len(sorted_probs):
            thresh_g = sorted_probs[n_positive]
        else:
            thresh_g = threshold
        y_pred_fair[mask] = (probs_g >= thresh_g).astype(int)
    
    return y_pred_fair


print("Funcoes de avaliacao e fairness definidas.")

## 6. Treinamento e Avaliação dos Modelos

Cada modelo será treinado e avaliado. Métricas armazenadas para comparação final.

In [ ]:
all_results = []
all_fairness = []
all_fairness_post = []
all_predictions = {}

### 6.1 Regressão Logística

In [ ]:
MODEL_NAME = "Logistic Regression"
print(f"{'='*60}")
print(f"Treinando: {MODEL_NAME}")
print(f"{'='*60}")

model = LogisticRegression(
    max_iter=1000, class_weight='balanced', solver='lbfgs', random_state=SEED, n_jobs=-1
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

results = evaluate_model(y_test.values, y_pred, y_prob, MODEL_NAME)
fairness_before = compute_fairness_metrics(y_test.values, y_pred, sens_test.values, MODEL_NAME)

y_pred_fair = fairness_postprocess(y_prob, sens_test.values, y_test.values)
fairness_after = compute_fairness_metrics(y_test.values, y_pred_fair, sens_test.values, MODEL_NAME)
results_fair = evaluate_model(y_test.values, y_pred_fair, y_prob, f"{MODEL_NAME} (Fair)")

all_results.append(results)
all_results.append(results_fair)
all_fairness.append(fairness_before)
all_fairness_post.append(fairness_after)
all_predictions[MODEL_NAME] = (y_pred, y_prob, y_pred_fair)

print(f"\n{MODEL_NAME} - Resultados:")
print(f"   F1: {results['F1-Score']:.4f} | AUC-ROC: {results['AUC-ROC']:.4f} | AUC-PR: {results['AUC-PR']:.4f}")
print(f"   DP Diff: {fairness_before['Demographic Parity Diff']:.4f} -> {fairness_after['Demographic Parity Diff']:.4f} (pos-proc.)")
print(classification_report(y_test, y_pred, target_names=['Legitima', 'Fraude']))

### 6.2 SGD Classifier (Modelo Linear Estocástico)

In [ ]:
MODEL_NAME = "SGD Classifier"
print(f"{'='*60}")
print(f"Treinando: {MODEL_NAME}")
print(f"{'='*60}")

model = CalibratedClassifierCV(
    SGDClassifier(loss='modified_huber', class_weight='balanced', max_iter=1000, random_state=SEED, n_jobs=-1),
    cv=3
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

results = evaluate_model(y_test.values, y_pred, y_prob, MODEL_NAME)
fairness_before = compute_fairness_metrics(y_test.values, y_pred, sens_test.values, MODEL_NAME)

y_pred_fair = fairness_postprocess(y_prob, sens_test.values, y_test.values)
fairness_after = compute_fairness_metrics(y_test.values, y_pred_fair, sens_test.values, MODEL_NAME)
results_fair = evaluate_model(y_test.values, y_pred_fair, y_prob, f"{MODEL_NAME} (Fair)")

all_results.append(results)
all_results.append(results_fair)
all_fairness.append(fairness_before)
all_fairness_post.append(fairness_after)
all_predictions[MODEL_NAME] = (y_pred, y_prob, y_pred_fair)

print(f"\n{MODEL_NAME} - Resultados:")
print(f"   F1: {results['F1-Score']:.4f} | AUC-ROC: {results['AUC-ROC']:.4f} | AUC-PR: {results['AUC-PR']:.4f}")
print(f"   DP Diff: {fairness_before['Demographic Parity Diff']:.4f} -> {fairness_after['Demographic Parity Diff']:.4f} (pos-proc.)")
print(classification_report(y_test, y_pred, target_names=['Legitima', 'Fraude']))

### 6.3 Random Forest

In [ ]:
MODEL_NAME = "Random Forest"
print(f"{'='*60}")
print(f"Treinando: {MODEL_NAME}")
print(f"{'='*60}")

model = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

results = evaluate_model(y_test.values, y_pred, y_prob, MODEL_NAME)
fairness_before = compute_fairness_metrics(y_test.values, y_pred, sens_test.values, MODEL_NAME)

y_pred_fair = fairness_postprocess(y_prob, sens_test.values, y_test.values)
fairness_after = compute_fairness_metrics(y_test.values, y_pred_fair, sens_test.values, MODEL_NAME)
results_fair = evaluate_model(y_test.values, y_pred_fair, y_prob, f"{MODEL_NAME} (Fair)")

all_results.append(results)
all_results.append(results_fair)
all_fairness.append(fairness_before)
all_fairness_post.append(fairness_after)
all_predictions[MODEL_NAME] = (y_pred, y_prob, y_pred_fair)

print(f"\n{MODEL_NAME} - Resultados:")
print(f"   F1: {results['F1-Score']:.4f} | AUC-ROC: {results['AUC-ROC']:.4f} | AUC-PR: {results['AUC-PR']:.4f}")
print(f"   DP Diff: {fairness_before['Demographic Parity Diff']:.4f} -> {fairness_after['Demographic Parity Diff']:.4f} (pos-proc.)")
print(classification_report(y_test, y_pred, target_names=['Legitima', 'Fraude']))

### 6.4 Extra Trees

In [ ]:
MODEL_NAME = "Extra Trees"
print(f"{'='*60}")
print(f"Treinando: {MODEL_NAME}")
print(f"{'='*60}")

model = ExtraTreesClassifier(
    n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

results = evaluate_model(y_test.values, y_pred, y_prob, MODEL_NAME)
fairness_before = compute_fairness_metrics(y_test.values, y_pred, sens_test.values, MODEL_NAME)

y_pred_fair = fairness_postprocess(y_prob, sens_test.values, y_test.values)
fairness_after = compute_fairness_metrics(y_test.values, y_pred_fair, sens_test.values, MODEL_NAME)
results_fair = evaluate_model(y_test.values, y_pred_fair, y_prob, f"{MODEL_NAME} (Fair)")

all_results.append(results)
all_results.append(results_fair)
all_fairness.append(fairness_before)
all_fairness_post.append(fairness_after)
all_predictions[MODEL_NAME] = (y_pred, y_prob, y_pred_fair)

print(f"\n{MODEL_NAME} - Resultados:")
print(f"   F1: {results['F1-Score']:.4f} | AUC-ROC: {results['AUC-ROC']:.4f} | AUC-PR: {results['AUC-PR']:.4f}")
print(f"   DP Diff: {fairness_before['Demographic Parity Diff']:.4f} -> {fairness_after['Demographic Parity Diff']:.4f} (pos-proc.)")
print(classification_report(y_test, y_pred, target_names=['Legitima', 'Fraude']))

### 6.5 SVM (LinearSVC com Calibração)

In [ ]:
MODEL_NAME = "Linear SVM"
print(f"{'='*60}")
print(f"Treinando: {MODEL_NAME}")
print(f"{'='*60}")

model = CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', max_iter=2000, random_state=SEED, dual='auto'),
    cv=3
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

results = evaluate_model(y_test.values, y_pred, y_prob, MODEL_NAME)
fairness_before = compute_fairness_metrics(y_test.values, y_pred, sens_test.values, MODEL_NAME)

y_pred_fair = fairness_postprocess(y_prob, sens_test.values, y_test.values)
fairness_after = compute_fairness_metrics(y_test.values, y_pred_fair, sens_test.values, MODEL_NAME)
results_fair = evaluate_model(y_test.values, y_pred_fair, y_prob, f"{MODEL_NAME} (Fair)")

all_results.append(results)
all_results.append(results_fair)
all_fairness.append(fairness_before)
all_fairness_post.append(fairness_after)
all_predictions[MODEL_NAME] = (y_pred, y_prob, y_pred_fair)

print(f"\n{MODEL_NAME} - Resultados:")
print(f"   F1: {results['F1-Score']:.4f} | AUC-ROC: {results['AUC-ROC']:.4f} | AUC-PR: {results['AUC-PR']:.4f}")
print(f"   DP Diff: {fairness_before['Demographic Parity Diff']:.4f} -> {fairness_after['Demographic Parity Diff']:.4f} (pos-proc.)")
print(classification_report(y_test, y_pred, target_names=['Legitima', 'Fraude']))

### 6.6 Rede Neural Leve — MLP (scikit-learn)

In [ ]:
MODEL_NAME = "MLP (sklearn)"
print(f"{'='*60}")
print(f"Treinando: {MODEL_NAME}")
print(f"{'='*60}")

model = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu', solver='adam', alpha=1e-4,
    batch_size=256, learning_rate='adaptive', learning_rate_init=1e-3,
    max_iter=100, random_state=SEED, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=10, verbose=False
)

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

results = evaluate_model(y_test.values, y_pred, y_prob, MODEL_NAME)
fairness_before = compute_fairness_metrics(y_test.values, y_pred, sens_test.values, MODEL_NAME)

y_pred_fair = fairness_postprocess(y_prob, sens_test.values, y_test.values)
fairness_after = compute_fairness_metrics(y_test.values, y_pred_fair, sens_test.values, MODEL_NAME)
results_fair = evaluate_model(y_test.values, y_pred_fair, y_prob, f"{MODEL_NAME} (Fair)")

all_results.append(results)
all_results.append(results_fair)
all_fairness.append(fairness_before)
all_fairness_post.append(fairness_after)
all_predictions[MODEL_NAME] = (y_pred, y_prob, y_pred_fair)

print(f"\n{MODEL_NAME} - Resultados:")
print(f"   F1: {results['F1-Score']:.4f} | AUC-ROC: {results['AUC-ROC']:.4f} | AUC-PR: {results['AUC-PR']:.4f}")
print(f"   DP Diff: {fairness_before['Demographic Parity Diff']:.4f} -> {fairness_after['Demographic Parity Diff']:.4f} (pos-proc.)")
print(classification_report(y_test, y_pred, target_names=['Legitima', 'Fraude']))

### 6.7 Rede Neural Densa Profunda (TensorFlow/Keras)

Arquitetura com múltiplas camadas densas, Batch Normalization, Dropout e class weights para lidar com desbalanceamento.

In [ ]:
MODEL_NAME = "Deep Neural Network (Keras)"
print(f"{'='*60}")
print(f"Treinando: {MODEL_NAME}")
print(f"{'='*60}")

class_weights_arr = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train.values)
class_weight_dict = {0: class_weights_arr[0], 1: class_weights_arr[1]}
print(f"Class weights: {class_weight_dict}")

n_features = X_train_scaled.shape[1]

def build_deep_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        # Bloco 1
        layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.4),
        # Bloco 2
        layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        # Bloco 3
        layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        # Bloco 4
        layers.Dense(32, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),
        # Saida
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

keras_model = build_deep_model(n_features)
keras_model.summary()

In [ ]:
keras_callbacks = [
    callbacks.EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max'),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

history = keras_model.fit(
    X_train_scaled, y_train.values,
    validation_split=0.15,
    epochs=100,
    batch_size=512,
    class_weight=class_weight_dict,
    callbacks=keras_callbacks,
    verbose=1
)

print("\nTreinamento concluido!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history.history['loss'], label='Treino', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validacao', linewidth=2)
axes[0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoca')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Treino', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validacao', linewidth=2)
axes[1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoca')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history.history['auc'], label='Treino', linewidth=2)
axes[2].plot(history.history['val_auc'], label='Validacao', linewidth=2)
axes[2].set_title('AUC-ROC', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoca')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Curvas de Treinamento - Deep Neural Network', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
y_prob_keras = keras_model.predict(X_test_scaled, verbose=0).flatten()
y_pred_keras = (y_prob_keras >= 0.5).astype(int)

results_keras = evaluate_model(y_test.values, y_pred_keras, y_prob_keras, MODEL_NAME)
fairness_keras = compute_fairness_metrics(y_test.values, y_pred_keras, sens_test.values, MODEL_NAME)

y_pred_keras_fair = fairness_postprocess(y_prob_keras, sens_test.values, y_test.values)
fairness_keras_post = compute_fairness_metrics(y_test.values, y_pred_keras_fair, sens_test.values, MODEL_NAME)
results_keras_post = evaluate_model(y_test.values, y_pred_keras_fair, y_prob_keras, f"{MODEL_NAME} (Fair)")

all_results.append(results_keras)
all_results.append(results_keras_post)
all_fairness.append(fairness_keras)
all_fairness_post.append(fairness_keras_post)
all_predictions[MODEL_NAME] = (y_pred_keras, y_prob_keras, y_pred_keras_fair)

print(f"\n{MODEL_NAME} - Resultados:")
print(f"   F1: {results_keras['F1-Score']:.4f} | AUC-ROC: {results_keras['AUC-ROC']:.4f} | AUC-PR: {results_keras['AUC-PR']:.4f}")
print(f"   DP Diff: {fairness_keras['Demographic Parity Diff']:.4f} -> {fairness_keras_post['Demographic Parity Diff']:.4f} (pos-proc.)")
print(classification_report(y_test, y_pred_keras, target_names=['Legitima', 'Fraude']))

## 7. Visualizações Comparativas

### 7.1 Curvas ROC e Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

colors_models = plt.cm.tab10(np.linspace(0, 1, len(all_predictions)))

ax = axes[0]
for idx, (name, (y_pred, y_prob, _)) in enumerate(all_predictions.items()):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.4f})', linewidth=2, color=colors_models[idx])

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Curvas ROC - Todos os Modelos', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)

ax = axes[1]
for idx, (name, (y_pred, y_prob, _)) in enumerate(all_predictions.items()):
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax.plot(rec, prec, label=f'{name} (AP={ap:.4f})', linewidth=2, color=colors_models[idx])

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Curvas Precision-Recall - Todos os Modelos', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.2 Matrizes de Confusão

In [ ]:
n_models = len(all_predictions)
n_row = 2
n_col = (n_models + 1) // 2
fig, axes = plt.subplots(n_row, n_col, figsize=(5 * n_col, 10))
axes = axes.flatten()

for idx, (name, (y_pred, y_prob, _)) in enumerate(all_predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[idx],
                xticklabels=['Legitima', 'Fraude'], yticklabels=['Legitima', 'Fraude'],
                linewidths=0.5)
    axes[idx].set_title(name, fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Real')
    axes[idx].set_xlabel('Predito')

for j in range(idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Matrizes de Confusao - Todos os Modelos', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 7.3 Comparativo de Métricas por Modelo

In [ ]:
base_results = [r for r in all_results if '(Fair)' not in r['Modelo']]
df_results = pd.DataFrame(base_results).set_index('Modelo')

fig, ax = plt.subplots(figsize=(14, 7))

metrics_to_plot = ['Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'AUC-PR']
x = np.arange(len(df_results))
width = 0.15
colors_bars = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, df_results[metric], width, label=metric,
           color=colors_bars[i], edgecolor='black', linewidth=0.5)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(df_results.index, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparativo de Metricas - Modelos Base', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left', bbox_to_anchor=(1, 1))
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Tabela Comparativa Final — Desempenho + Fairness

### 8.1 Desempenho dos Modelos (sem pós-processamento)

In [ ]:
base_results = [r for r in all_results if '(Fair)' not in r['Modelo']]
df_perf = pd.DataFrame(base_results).set_index('Modelo')

for col in df_perf.columns:
    df_perf[col] = df_perf[col].apply(lambda x: round(x, 4) if isinstance(x, float) else x)

print("TABELA DE DESEMPENHO - MODELOS BASE\n")
print(df_perf.to_string())

### 8.2 Métricas de Fairness — Antes do Pós-processamento

In [ ]:
df_fair_before = pd.DataFrame(all_fairness).set_index('Modelo')
for col in df_fair_before.columns:
    df_fair_before[col] = df_fair_before[col].apply(lambda x: round(x, 4) if isinstance(x, float) else x)

print("FAIRNESS - ANTES DO POS-PROCESSAMENTO\n")
print(df_fair_before.to_string())

### 8.3 Métricas de Fairness — Depois do Pós-processamento

In [ ]:
df_fair_after = pd.DataFrame(all_fairness_post).set_index('Modelo')
for col in df_fair_after.columns:
    df_fair_after[col] = df_fair_after[col].apply(lambda x: round(x, 4) if isinstance(x, float) else x)

print("FAIRNESS - DEPOIS DO POS-PROCESSAMENTO\n")
print(df_fair_after.to_string())

### 8.4 Tabela Consolidada: Desempenho + Fairness

In [ ]:
consolidated = []
model_names = [r['Modelo'] for r in all_fairness]

for i, name in enumerate(model_names):
    base = [r for r in all_results if r['Modelo'] == name][0]
    fair_b = all_fairness[i]
    fair_a = all_fairness_post[i]
    
    consolidated.append({
        'Modelo': name,
        'F1': round(base['F1-Score'], 4),
        'AUC-ROC': round(base['AUC-ROC'], 4),
        'AUC-PR': round(base['AUC-PR'], 4),
        'Recall': round(base['Recall'], 4),
        'Precision': round(base['Precision'], 4),
        'DP Diff (antes)': round(fair_b['Demographic Parity Diff'], 4),
        'DP Diff (depois)': round(fair_a['Demographic Parity Diff'], 4),
        'EO TPR Diff (antes)': round(fair_b['Equalized Odds Diff (TPR)'], 4),
        'EO TPR Diff (depois)': round(fair_a['Equalized Odds Diff (TPR)'], 4),
    })

df_consolidated = pd.DataFrame(consolidated).set_index('Modelo')

print("=" * 100)
print("TABELA CONSOLIDADA - DESEMPENHO + FAIRNESS")
print("=" * 100)
print()
print(df_consolidated.to_string())
print()
print("Legenda:")
print("  DP Diff = Demographic Parity Difference (menor = mais justo)")
print("  EO TPR Diff = Equalized Odds Difference no TPR (menor = mais justo)")
print("  'antes' = sem pos-processamento | 'depois' = com ajuste de threshold por grupo")

### 8.5 Visualização da Tabela Consolidada

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

models_list = df_consolidated.index.tolist()
x = np.arange(len(models_list))
w = 0.25

ax = axes[0]
ax.bar(x - w, df_consolidated['F1'], w, label='F1-Score', color='#3498db', edgecolor='black')
ax.bar(x, df_consolidated['AUC-ROC'], w, label='AUC-ROC', color='#2ecc71', edgecolor='black')
ax.bar(x + w, df_consolidated['AUC-PR'], w, label='AUC-PR', color='#e74c3c', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=25, ha='right', fontsize=9)
ax.set_title('Metricas de Desempenho', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
w2 = 0.2
ax.bar(x - w2, df_consolidated['DP Diff (antes)'], w2, label='DP Diff (antes)', color='#e74c3c', edgecolor='black')
ax.bar(x, df_consolidated['DP Diff (depois)'], w2, label='DP Diff (depois)', color='#2ecc71', edgecolor='black')
ax.bar(x + w2, df_consolidated['EO TPR Diff (antes)'], w2, label='EO TPR (antes)', color='#f39c12', edgecolor='black', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=25, ha='right', fontsize=9)
ax.set_title('Metricas de Fairness (menor = mais justo)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparativo Final - Desempenho vs Fairness', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 9. Importância das Features (Modelos Baseados em Árvore)

In [ ]:
# Recuperar modelos de arvore do historico
# Os modelos foram salvos como 'model' nas celulas anteriores
# Precisamos re-referenciar. Vamos usar os objetos globais.

# Random Forest e Extra Trees foram os ultimos a usar 'model'
# Vamos retreinar rapidamente apenas para feature importance

rf_fi = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_fi.fit(X_train_scaled, y_train)

et_fi = ExtraTreesClassifier(
    n_estimators=200, max_depth=15, min_samples_split=10, min_samples_leaf=5,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
et_fi.fit(X_train_scaled, y_train)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for idx, (name, mdl) in enumerate([("Random Forest", rf_fi), ("Extra Trees", et_fi)]):
    ax = axes[idx]
    importances = mdl.feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)
    top_20 = feat_imp.tail(20)
    
    top_20.plot(kind='barh', ax=ax, color='#3498db', edgecolor='black')
    ax.set_title(f'Top 20 Features - {name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importancia')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance - Modelos de Arvore', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Conclusão

Este notebook implementou e comparou **7 modelos** de diferentes famílias para detecção de fraude bancária usando o dataset BAF (NeurIPS 2022):

1. **Modelos Lineares:** Logistic Regression e SGD Classifier — baselines eficientes e interpretáveis.
2. **Modelos de Agregação (Bagging):** Random Forest e Extra Trees — capturam relações não-lineares com estabilidade.
3. **SVM:** LinearSVC com calibração — fronteira de decisão linear em alta dimensionalidade.
4. **Rede Neural Leve:** MLP (scikit-learn) — aprendizado de representações não-lineares com custo moderado.
5. **Rede Neural Profunda:** Keras/TensorFlow com BatchNorm, Dropout e regularização L2 — maior capacidade.

### Fairness
O pós-processamento por ajuste de threshold por grupo visa equalizar as taxas de predição positiva entre grupos protegidos (Demographic Parity). A tabela consolidada permite analisar o trade-off entre desempenho e justiça algorítmica.

### Reprodutibilidade
- Seed fixa (`SEED=42`) em todos os modelos
- Splits estratificados
- Class weights balanceados

---
*Notebook gerado para fins acadêmicos — Benchmark de Detecção de Fraude em Contas Bancárias (BAF Dataset)*